In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy.stats import mannwhitneyu
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [2]:
EMBEDDING_DIM   = 100
MARGIN          = 1.0
LR              = 0.01
EPOCHS          = 30          # per temporal window
BATCH_SIZE      = 2048
NORM            = 1           # L1 distance (standard for TransE)
NEG_SAMPLES     = 1           # negatives per positive
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_PATH = '../../outputs'

EDGES_PATH = f'{BASE_PATH}/final/knowledge_edges.csv'
PAPERS_PATH = f'{BASE_PATH}/final/paper_nodes.csv'

In [5]:
edges = pd.read_csv(EDGES_PATH)
papers = pd.read_csv(PAPERS_PATH)

edges = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)

paper_split = dict(zip(papers['node_id'], papers['split']))
paper_year = dict(zip(papers['node_id'], papers['year']))

edges['split'] = edges['source'].map(paper_split)

skg_edges = edges[edges['split'] == 'SKG'].copy()
novel_edges = edges[edges['split'] == 'NOVEL'].copy()

print(f"SKG edges: {len(skg_edges)}")
print(f"NOVEL edges: {len(novel_edges)}")

SKG edges: 378527
NOVEL edges: 82599


In [6]:
all_entities = pd.unique(pd.concat([edges['source'], edges['target']]))
all_relations = pd.unique(edges['predicate'])

entity2id = {e : i for i , e in enumerate(all_entities)}
relation2id = {r : i for i , r in enumerate(all_relations)}

NUM_ENTITIES = len(entity2id)
NUM_RELATIONS = len(relation2id)

print(f"Entities: {NUM_ENTITIES:,}")
print(f"Relations: {NUM_RELATIONS:,}")

Entities: 183,693
Relations: 18,494


In [ ]:
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations , dim , norm=1):
        super().__init__()
        self.norm = norm 
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)

        nn.init.uniform_(self.entity_emb.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))
        
        nn.init.uniform_(self.relation_emb.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))

        self.relation_emb.weight.data = nn.functional.normalize(self.relation_emb.weight.data, p=2, dim=1 )
        
    
    def forward(self, heads, relations, tails):
        h = nn.functional.normalize(self.entity_emb(heads), p=2 , dim= 1)
        r = self.relation_emb(relations)
        t = nn.functional.normalize(self.entity_emb(tails), p=2, dim=1 )
        # Dist : ||h + r - t||
        score = torch.norm(h+r-t, p = self.norm, dim=1)
        return score

    def score_triples(self, heads, relations, tails):
        """Returns novelty score = distance (higher = more novel)"""
        with torch.no_grad():
            return self.forward(heads, relations, tails).cpu().numpy()


In [16]:
class TripleDataset(Dataset):
    def __init__(self, triples_df, entity2id, relation2id, all_entity_ids, neg_samples=1):
        self.entity2id = entity2id
        self.relation2id = relation2id
        self.all_entity_ids = np.array(list(all_entity_ids.values()))
        self.neg_samples = neg_samples

        valid = (
            triples_df['source'].isin(entity2id) &
            triples_df['target'].isin(entity2id) &
            triples_df['predicate'].isin(relation2id)
        )

        df = triples_df[valid].reset_index(drop=True)
        self.heads = df['source'].map(entity2id).values
        self.relations = df['predicate'].map(relation2id).values
        self.tails = df['target'].map(entity2id).values

    def __len__(self):
        return len(self.heads)

    def __getitem__(self, idx):
        h, r, t = self.heads[idx], self.relations[idx], self.tails[idx]
        neg_t = np.random.choice(self.all_entity_ids)
        return (
            torch.tensor(h, dtype=torch.long),
            torch.tensor(r, dtype=torch.long),
            torch.tensor(t, dtype=torch.long),
            torch.tensor(neg_t, dtype=torch.long),
        )
    
def margin_loss(pos_score, neg_score, margin):
    return torch.clamp(margin + pos_score - neg_score, min = 0.0).mean()


In [17]:
novel_years = sorted(novel_edges['year'].unique())
print(f"\nNovel years to score: {novel_years}")


Novel years to score: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [18]:
results = []

for T in novel_years:
    print(f"\n{'='*50}")
    print(f"  Year T = {T}")

    train_df = skg_edges[skg_edges['year'] < T].copy()
    print(f" Training triples (SKG year < {T}):  {len(train_df):,}")

    if len(train_df) < 100:
        print(f"  ⚠ Too few training triples — skipping year {T}")
        continue

    model = TransE(NUM_ENTITIES, NUM_RELATIONS, EMBEDDING_DIM, norm=NORM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr = LR)

    dataset = TripleDataset(train_df, entity2id, relation2id, entity2id, neg_samples=NEG_SAMPLES)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last= False)

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0.0 
        for h , r, t , neg_t in loader:
            h, r, t, neg_t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE), neg_t.to(DEVICE)
            optimizer.zero_grad()
            pos_score = model(h, r, t)
            neg_score = model(h, r, neg_t)
            loss = margin_loss(pos_score, neg_score, MARGIN)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f" Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(loader):.4f}")

    model.eval()
    novel_at_T = novel_edges[novel_edges['year'] == T]
    novel_papers_at_T = novel_at_T['source'].unique()
    print(f"   Scoring {len(novel_papers_at_T):,} Novel papers at year {T}")

    for paper_id in novel_papers_at_T:
        paper_triples = novel_at_T[novel_at_T['source'] == paper_id]

        valid = (
            paper_triples['source'].isin(entity2id) &
            paper_triples['target'].isin(entity2id) &
            paper_triples['predicate'].isin(relation2id)
        
        )

        paper_triples = paper_triples[valid]

        if len(paper_triples) == 0:
            continue 

        h_idx = torch.tensor(paper_triples['source'].map(entity2id).values, dtype=torch.long).to(DEVICE)
        r_idx = torch.tensor(paper_triples['predicate'].map(relation2id).values, dtype=torch.long).to(DEVICE)
        t_idx = torch.tensor(paper_triples['target'].map(entity2id).values, dtype=torch.long).to(DEVICE)

        scores = model.score_triples(h_idx, r_idx, t_idx)
        novelty_score = float(np.mean(scores))

        results.append({
            'paper_id' : paper_id,
            'transE_score' : novelty_score,
            'year' : T,
            'num_triples':len(paper_triples)
        })

print(f'\n{"="*50}')
print(f"Scored {len(results)} NOVEL papers total")



  Year T = 2020
 Training triples (SKG year < 2020):  169,904
 Epoch 10/30 - Loss: 0.0165
 Epoch 20/30 - Loss: 0.0105
 Epoch 30/30 - Loss: 0.0090
   Scoring 4 Novel papers at year 2020

  Year T = 2021
 Training triples (SKG year < 2021):  234,348
 Epoch 10/30 - Loss: 0.0182
 Epoch 20/30 - Loss: 0.0129
 Epoch 30/30 - Loss: 0.0106
   Scoring 404 Novel papers at year 2021

  Year T = 2022
 Training triples (SKG year < 2022):  234,348
 Epoch 10/30 - Loss: 0.0185
 Epoch 20/30 - Loss: 0.0123
 Epoch 30/30 - Loss: 0.0107
   Scoring 23 Novel papers at year 2022

  Year T = 2023
 Training triples (SKG year < 2023):  298,455
 Epoch 10/30 - Loss: 0.0192
 Epoch 20/30 - Loss: 0.0134
 Epoch 30/30 - Loss: 0.0112
   Scoring 86 Novel papers at year 2023

  Year T = 2024
 Training triples (SKG year < 2024):  366,526
 Epoch 10/30 - Loss: 0.0197
 Epoch 20/30 - Loss: 0.0138
 Epoch 30/30 - Loss: 0.0109
   Scoring 9 Novel papers at year 2024

  Year T = 2025
 Training triples (SKG year < 2025):  377,828
 Ep

In [19]:
results_df = pd.DataFrame(results)
results_df.to_csv(f'{BASE_PATH}/final/TransE_novelty_scores.csv', index=False)
print(results_df.head(10))

       paper_id  transE_score  year  num_triples
0    NOVEL_MT_9     14.483563  2020           53
1   NOVEL_MT_95     15.739375  2020          181
2  NOVEL_MT_163     15.571033  2020          194
3   NOVEL_QA_38     15.366490  2020           84
4   NOVEL_DIA_0     17.467312  2021           99
5   NOVEL_DIA_1     16.704102  2021          132
6   NOVEL_DIA_2     16.513563  2021           94
7   NOVEL_DIA_3     16.539328  2021          206
8   NOVEL_DIA_4     17.182657  2021          206
9   NOVEL_DIA_5     16.521978  2021          103


In [20]:
print(f"\n-- Scoring SKG papers for validation --")
skg_papers = papers[papers['split'] == 'SKG']
skg_results = []

T_val = max(novel_years) + 1
train_df = skg_edges[skg_edges['year'] < T_val].copy()
model_val = TransE(NUM_ENTITIES, NUM_RELATIONS, EMBEDDING_DIM, norm=NORM).to(DEVICE)
opt_val = torch.optim.Adam(model_val.parameters(), lr = LR)
dataset_val = TripleDataset(train_df, entity2id, relation2id, entity2id, neg_samples=NEG_SAMPLES)
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

model_val.train()
for epoch in range(EPOCHS):
    for h, r,t , neg_t in loader_val:
        h,r,t,neg_t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE), neg_t.to(DEVICE)
        opt_val.zero_grad()
        loss = margin_loss(model_val(h,r,t), model_val(h,r,neg_t), MARGIN)
        loss.backward()
        opt_val.step()

model_val.eval()
for _, row in skg_papers.iterrows():
    pid = row['node_id']
    yr = row['year']
    paper_triples = skg_edges[skg_edges['source'] == pid]
    valid = (
        paper_triples['source'].isin(entity2id) &
        paper_triples['target'].isin(entity2id) &
        paper_triples['predicate'].isin(relation2id)
    )
    paper_triples = paper_triples[valid]
    if len(paper_triples) == 0:
        continue
    h_idx = torch.tensor(paper_triples['source'].map(entity2id).values, dtype=torch.long).to(DEVICE)
    r_idx = torch.tensor(paper_triples['predicate'].map(relation2id).values, dtype=torch.long).to(DEVICE)
    t_idx = torch.tensor(paper_triples['target'].map(entity2id).values, dtype=torch.long).to(DEVICE)
    scores = float(np.mean(model_val.score_triples(h_idx, r_idx, t_idx)))
    skg_results.append({'paper_id': pid, 'transE_score': scores, 'year': yr})

skg_df = pd.DataFrame(skg_results)



-- Scoring SKG papers for validation --


In [21]:
# Mann-Whitney U test
novel_scores = results_df['transE_score'].values
skg_scores   = skg_df['transE_score'].values

stat, p = mannwhitneyu(novel_scores, skg_scores, alternative='greater')
n1, n2  = len(novel_scores), len(skg_scores)
r_effect = stat / (n1 * n2)

print(f"\n── Validation Results ──")
print(f"NOVEL mean score: {novel_scores.mean():.4f}  (n={n1})")
print(f"SKG   mean score: {skg_scores.mean():.4f}  (n={n2})")
print(f"Mann-Whitney U={stat:.1f}, p={p:.4f}, r={r_effect:.3f}")
if p < 0.05 and novel_scores.mean() > skg_scores.mean():
    print("✅ SUCCESS: NOVEL > SKG, p < 0.05")
else:
    print("❌ Did not meet success criteria — check score direction or weights")


── Validation Results ──
NOVEL mean score: 17.9796  (n=528)
SKG   mean score: 19.3611  (n=2543)
Mann-Whitney U=390332.0, p=1.0000, r=0.291
❌ Did not meet success criteria — check score direction or weights


In [3]:
edges  = pd.read_csv(EDGES_PATH)
papers = pd.read_csv(PAPERS_PATH)

edges = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)

paper_split = dict(zip(papers['node_id'], papers['split']))
paper_year  = dict(zip(papers['node_id'], papers['year']))
edges['split'] = edges['source'].map(paper_split)

skg_edges   = edges[edges['split'] == 'SKG'].copy()
novel_edges = edges[edges['split'] == 'NOVEL'].copy()

print(f"SKG edges:   {len(skg_edges):,}")
print(f"NOVEL edges: {len(novel_edges):,}")

SKG edges:   378,527
NOVEL edges: 82,599


In [4]:
all_entities  = pd.unique(edges['target'])          # only target entities
all_relations = pd.unique(edges['predicate'])

entity2id   = {e: i for i, e in enumerate(all_entities)}
relation2id = {r: i for i, r in enumerate(all_relations)}

NUM_ENTITIES  = len(entity2id)
NUM_RELATIONS = len(relation2id)

print(f"Entities (targets only): {NUM_ENTITIES:,}")
print(f"Relations:               {NUM_RELATIONS:,}")

Entities (targets only): 180,291
Relations:               18,494


In [5]:
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, dim, norm=1):
        super().__init__()
        self.norm = norm
        self.dim  = dim
        self.entity_emb   = nn.Embedding(num_entities,  dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.uniform_(self.entity_emb.weight,  -6/np.sqrt(dim), 6/np.sqrt(dim))
        nn.init.uniform_(self.relation_emb.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))
        self.relation_emb.weight.data = nn.functional.normalize(
            self.relation_emb.weight.data, p=2, dim=1)

    def score(self, heads_emb, relations, tails):
        """
        heads_emb : pre-computed head embeddings [B, dim]
        relations : relation indices [B]
        tails     : tail entity indices [B]
        """
        r = self.relation_emb(relations)
        t = nn.functional.normalize(self.entity_emb(tails), p=2, dim=1)
        h = nn.functional.normalize(heads_emb, p=2, dim=1)
        return torch.norm(h + r - t, p=self.norm, dim=1)

    def forward(self, head_indices, relations, tails):
        """Standard forward for training — head_indices are entity indices"""
        h_emb = self.entity_emb(head_indices)
        return self.score(h_emb, relations, tails)

    def get_mean_head_for_relations(self, relation_to_head_indices):
        """
        Returns mean head embedding per relation.
        relation_to_head_indices: dict {relation_id: [entity_ids that appear as heads with this relation]}
        Returns: dict {relation_id: mean_embedding tensor}
        """
        mean_heads = {}
        with torch.no_grad():
            for rel_id, head_ids in relation_to_head_indices.items():
                if len(head_ids) == 0:
                    mean_heads[rel_id] = torch.zeros(self.dim).to(DEVICE)
                else:
                    h_indices = torch.tensor(head_ids, dtype=torch.long).to(DEVICE)
                    embs = nn.functional.normalize(
                        self.entity_emb(h_indices), p=2, dim=1)
                    mean_heads[rel_id] = embs.mean(dim=0)
        return mean_heads

In [6]:
class TripleDataset(Dataset):
    """
    KEY CHANGE: heads are now target entities that appear as tails in other triples.
    We train on (tail_entity_1, relation, tail_entity_2) co-occurrence patterns.

    Since all our triples are (paper_id → entity), we model entity co-occurrence:
    Two entities co-occurring under the same paper via same/different relations
    forms an implicit entity-entity association we can score.

    Simpler approach: use the same (target, predicate, target) structure
    where head = a randomly sampled entity that appeared with this predicate in SKG.
    """
    def __init__(self, triples_df, entity2id, relation2id, neg_samples=1):
        self.entity2id   = entity2id
        self.relation2id = relation2id
        self.neg_samples = neg_samples
        self.all_entity_arr = np.array(list(entity2id.values()), dtype=np.int64)

        # Filter to known vocab (targets and predicates only)
        valid = (
            triples_df['target'].isin(entity2id) &
            triples_df['predicate'].isin(relation2id)
        )
        df = triples_df[valid].reset_index(drop=True)

        # Build relation → list of tail entity indices seen with that relation
        # We'll use these as "representative heads" for the relation
        rel_to_tails = defaultdict(list)
        for _, row in df.iterrows():
            r_id = relation2id[row['predicate']]
            t_id = entity2id[row['target']]
            rel_to_tails[r_id].append(t_id)

        # For training: head = a tail entity associated with this relation
        # (entity acting as proxy head), relation = predicate, tail = target
        # This teaches the model what (relation, tail) combinations are valid
        self.heads     = []
        self.relations = []
        self.tails     = []

        for _, row in df.iterrows():
            r_id = relation2id[row['predicate']]
            t_id = entity2id[row['target']]
            # Use a random entity seen with this relation as the head proxy
            candidate_heads = rel_to_tails[r_id]
            h_id = candidate_heads[np.random.randint(len(candidate_heads))]
            self.heads.append(h_id)
            self.relations.append(r_id)
            self.tails.append(t_id)

        self.heads     = np.array(self.heads,     dtype=np.int64)
        self.relations = np.array(self.relations, dtype=np.int64)
        self.tails     = np.array(self.tails,     dtype=np.int64)

        # Also store relation → head indices for mean-head computation later
        self.rel_to_head_ids = {k: list(set(v)) for k, v in rel_to_tails.items()}

    def __len__(self):
        return len(self.heads)

    def __getitem__(self, idx):
        h, r, t = self.heads[idx], self.relations[idx], self.tails[idx]
        neg_tails = np.random.choice(self.all_entity_arr, size=self.neg_samples)
        return (
            torch.tensor(h,         dtype=torch.long),
            torch.tensor(r,         dtype=torch.long),
            torch.tensor(t,         dtype=torch.long),
            torch.tensor(neg_tails, dtype=torch.long),
        )


def margin_loss(pos_score, neg_scores, margin):
    # neg_scores: [B, neg_samples]
    losses = torch.clamp(margin + pos_score.unsqueeze(1) - neg_scores, min=0.0)
    return losses.mean()

In [7]:
novel_years = sorted(novel_edges['year'].unique())
print(f"\nNOVEL years to score: {novel_years}")

results = []

for T in novel_years:
    print(f"\n{'='*55}")
    print(f"  Year T = {T}")

    train_df = skg_edges[skg_edges['year'] < T].copy()
    print(f"  Training triples (SKG year < {T}): {len(train_df):,}")

    if len(train_df) < 100:
        print(f"  ⚠ Too few training triples — skipping")
        continue

    model     = TransE(NUM_ENTITIES, NUM_RELATIONS, EMBEDDING_DIM, norm=NORM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    dataset = TripleDataset(train_df, entity2id, relation2id, NEG_SAMPLES)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=0, drop_last=False)

    # Train
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for h, r, t, neg_t in loader:
            h    = h.to(DEVICE)
            r    = r.to(DEVICE)
            t    = t.to(DEVICE)
            neg_t = neg_t.to(DEVICE)          # [B, neg_samples]

            optimizer.zero_grad()

            # Get head embeddings
            h_emb = model.entity_emb(h)

            pos_score = model.score(h_emb, r, t)

            # Score all negatives
            B = h.shape[0]
            neg_scores_list = []
            for k in range(NEG_SAMPLES):
                ns = model.score(h_emb, r, neg_t[:, k])
                neg_scores_list.append(ns.unsqueeze(1))
            neg_scores = torch.cat(neg_scores_list, dim=1)

            loss = margin_loss(pos_score, neg_scores, MARGIN)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{EPOCHS}  loss={total_loss/len(loader):.4f}")

    # ── Score NOVEL papers at year T ──────────────────────────────────────────
    # KEY FIX: use mean-head scoring
    # For each (predicate, tail) pair in a NOVEL paper:
    #   mean_head_r = mean embedding of all entities seen as tails with predicate r in training
    #   novelty_contribution = ||mean_head_r + r_emb - t_emb||
    # Paper score = mean over all triples

    model.eval()

    # Pre-compute mean head embeddings per relation from training data
    mean_heads = model.get_mean_head_for_relations(dataset.rel_to_head_ids)

    novel_at_T        = novel_edges[novel_edges['year'] == T]
    novel_papers_at_T = novel_at_T['source'].unique()
    print(f"  Scoring {len(novel_papers_at_T)} NOVEL papers at year {T}")

    for paper_id in novel_papers_at_T:
        paper_triples = novel_at_T[novel_at_T['source'] == paper_id]

        # Filter to known vocab (targets + predicates)
        valid = (
            paper_triples['target'].isin(entity2id) &
            paper_triples['predicate'].isin(relation2id)
        )
        paper_triples = paper_triples[valid]

        if len(paper_triples) == 0:
            # Fallback score: use global mean (all unknown triples = max surprise)
            results.append({
                'paper_id':       paper_id,
                'transE_score':   float('nan'),
                'year':           T,
                'num_triples':    0
            })
            continue

        triple_scores = []
        for _, row in paper_triples.iterrows():
            r_id = relation2id[row['predicate']]
            t_id = entity2id[row['target']]

            # Get mean head for this relation
            if r_id in mean_heads:
                h_emb_mean = mean_heads[r_id].unsqueeze(0)   # [1, dim]
            else:
                # Relation never seen in training — maximum novelty
                # Use zero vector (furthest from trained space)
                h_emb_mean = torch.zeros(1, EMBEDDING_DIM).to(DEVICE)

            r_tensor = torch.tensor([r_id], dtype=torch.long).to(DEVICE)
            t_tensor = torch.tensor([t_id], dtype=torch.long).to(DEVICE)

            with torch.no_grad():
                score = model.score(h_emb_mean, r_tensor, t_tensor)
            triple_scores.append(score.item())

        novelty_score = float(np.mean(triple_scores))

        results.append({
            'paper_id':     paper_id,
            'transE_score': novelty_score,
            'year':         T,
            'num_triples':  len(paper_triples)
        })

print(f"\n{'='*55}")
print(f"Scored {len(results)} NOVEL papers total")


NOVEL years to score: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

  Year T = 2020
  Training triples (SKG year < 2020): 169,904
    Epoch 10/30  loss=0.0298
    Epoch 20/30  loss=0.0190
    Epoch 30/30  loss=0.0091
  Scoring 4 NOVEL papers at year 2020

  Year T = 2021
  Training triples (SKG year < 2021): 234,348
    Epoch 10/30  loss=0.0340
    Epoch 20/30  loss=0.0220
    Epoch 30/30  loss=0.0108
  Scoring 404 NOVEL papers at year 2021

  Year T = 2022
  Training triples (SKG year < 2022): 234,348
    Epoch 10/30  loss=0.0342
    Epoch 20/30  loss=0.0216
    Epoch 30/30  loss=0.0110
  Scoring 23 NOVEL papers at year 2022

  Year T = 2023
  Training triples (SKG year < 2023): 298,455
    Epoch 10/30  loss=0.0315
    Epoch 20/30  loss=0.0214
    Epoch 30/30  loss=0.0098
  Scoring 86 NOVEL papers at year 2023

  Year T = 2024
  Training triples (SKG year < 2024): 366,526
    Epoch 10/30  loss=0.0301
    Epoch 20/30  loss=0.0203
   

In [8]:
results_df = pd.DataFrame(results)
results_df_clean = results_df.dropna(subset=['transE_score'])
results_df_clean.to_csv(f'{BASE_PATH}/final/TransE_novelty_scores.csv', index=False)
print(f"Saved → {BASE_PATH}/final/TransE_novelty_scores.csv")
print(results_df_clean.head(10))

Saved → ../../outputs/final/TransE_novelty_scores.csv
       paper_id  transE_score  year  num_triples
0    NOVEL_MT_9     19.068465  2020           53
1   NOVEL_MT_95     20.177295  2020          181
2  NOVEL_MT_163     20.056812  2020          194
3   NOVEL_QA_38     19.667933  2020           84
4   NOVEL_DIA_0     21.925641  2021           99
5   NOVEL_DIA_1     21.223064  2021          132
6   NOVEL_DIA_2     20.642224  2021           94
7   NOVEL_DIA_3     21.479903  2021          206
8   NOVEL_DIA_4     22.206897  2021          206
9   NOVEL_DIA_5     21.593967  2021          103


In [9]:
print("\n── Scoring SKG papers for validation ──")
T_val    = max(novel_years) + 1
train_df_full = skg_edges.copy()

model_val = TransE(NUM_ENTITIES, NUM_RELATIONS, EMBEDDING_DIM, norm=NORM).to(DEVICE)
opt_val   = torch.optim.Adam(model_val.parameters(), lr=LR)
dataset_val = TripleDataset(train_df_full, entity2id, relation2id, NEG_SAMPLES)
loader_val  = DataLoader(dataset_val, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=0)

model_val.train()
for epoch in range(EPOCHS):
    for h, r, t, neg_t in loader_val:
        h, r, t, neg_t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE), neg_t.to(DEVICE)
        opt_val.zero_grad()
        h_emb = model_val.entity_emb(h)
        pos_score = model_val.score(h_emb, r, t)
        neg_scores_list = [model_val.score(h_emb, r, neg_t[:, k]).unsqueeze(1)
                           for k in range(NEG_SAMPLES)]
        neg_scores = torch.cat(neg_scores_list, dim=1)
        loss = margin_loss(pos_score, neg_scores, MARGIN)
        loss.backward()
        opt_val.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Validation model epoch {epoch+1}/{EPOCHS}")

model_val.eval()
mean_heads_val = model_val.get_mean_head_for_relations(dataset_val.rel_to_head_ids)

skg_results = []
for paper_id in skg_edges['source'].unique():
    pt = skg_edges[skg_edges['source'] == paper_id]
    valid = pt['target'].isin(entity2id) & pt['predicate'].isin(relation2id)
    pt = pt[valid]
    if len(pt) == 0:
        continue
    scores = []
    for _, row in pt.iterrows():
        r_id = relation2id[row['predicate']]
        t_id = entity2id[row['target']]
        h_emb_mean = mean_heads_val.get(
            r_id, torch.zeros(1, EMBEDDING_DIM).to(DEVICE))
        if h_emb_mean.dim() == 1:
            h_emb_mean = h_emb_mean.unsqueeze(0)
        r_t = torch.tensor([r_id], dtype=torch.long).to(DEVICE)
        t_t = torch.tensor([t_id], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            s = model_val.score(h_emb_mean, r_t, t_t)
        scores.append(s.item())
    skg_results.append({'paper_id': paper_id,
                        'transE_score': float(np.mean(scores))})

skg_df = pd.DataFrame(skg_results)




novel_scores = results_df_clean['transE_score'].values
skg_scores   = skg_df['transE_score'].values

stat, p = mannwhitneyu(novel_scores, skg_scores, alternative='greater')
n1, n2  = len(novel_scores), len(skg_scores)
r_eff   = stat / (n1 * n2)

print(f"\n── Validation Results ──")
print(f"NOVEL mean: {novel_scores.mean():.4f}  (n={n1})")
print(f"SKG   mean: {skg_scores.mean():.4f}  (n={n2})")
print(f"Mann-Whitney U={stat:.1f},  p={p:.6f},  r={r_eff:.3f}")

if p < 0.05 and novel_scores.mean() > skg_scores.mean():
    print("✅ SUCCESS: NOVEL > SKG, p < 0.05")
else:
    print("❌ Did not pass — see direction and p-value above")
    # Diagnostic: try reversed direction
    stat2, p2 = mannwhitneyu(novel_scores, skg_scores, alternative='less')
    r2 = stat2 / (n1 * n2)
    print(f"\n   Reversed test (NOVEL < SKG): p={p2:.6f}, r={r2:.3f}")
    print("   If this passes, scoring is still inverted — report this.")


── Scoring SKG papers for validation ──
  Validation model epoch 10/30
  Validation model epoch 20/30
  Validation model epoch 30/30

── Validation Results ──
NOVEL mean: 22.1371  (n=528)
SKG   mean: 26.5933  (n=2543)
Mann-Whitney U=120052.0,  p=1.000000,  r=0.089
❌ Did not pass — see direction and p-value above

   Reversed test (NOVEL < SKG): p=0.000000, r=0.089
   If this passes, scoring is still inverted — report this.


In [10]:
edges  = pd.read_csv(EDGES_PATH)
papers = pd.read_csv(PAPERS_PATH)

edges = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)

paper_split = dict(zip(papers['node_id'], papers['split']))
edges['split'] = edges['source'].map(paper_split)

skg_edges   = edges[edges['split'] == 'SKG'].copy()
novel_edges = edges[edges['split'] == 'NOVEL'].copy()

print(f"SKG edges:   {len(skg_edges):,}")
print(f"NOVEL edges: {len(novel_edges):,}")


SKG edges:   378,527
NOVEL edges: 82,599


In [11]:
def compute_pe_novelty(paper_triples, pe_count, pe_paper_set,
                       num_skg_papers, num_skg_triples):
    """
    Three complementary signals per (predicate, entity) pair:

    1. Pair frequency novelty  — how rarely has this (pred, entity) been used?
       score = 1 / (1 + log(1 + count))
       Unseen pair → score = 1.0   |   Very common pair → score → 0

    2. Entity specificity      — is this entity only used with a few relations?
       Broad entities (appears with many predicates) are less informative.
       score = 1 / (1 + log(1 + num_relations_for_entity))

    3. Predicate rarity        — is this predicate itself rarely used in SKG?
       Rare predicates = more specialised = more novel contribution.
       score = 1 / (1 + log(1 + predicate_count))

    Final per-triple score = geometric mean of all three signals.
    Paper score = mean across all triples.
    """
    triple_scores = []

    for _, row in paper_triples.iterrows():
        pred   = row['predicate']
        entity = row['target']
        key    = (pred, entity)

        # Signal 1: (predicate, entity) pair frequency
        pair_count = pe_count.get(key, 0)
        sig1 = 1.0 / (1.0 + np.log1p(pair_count))

        # Signal 2: entity specificity
        entity_pred_count = pe_paper_set['entity_pred_count'].get(entity, 0)
        sig2 = 1.0 / (1.0 + np.log1p(entity_pred_count))

        # Signal 3: predicate rarity
        pred_count = pe_paper_set['pred_count'].get(pred, 0)
        sig3 = 1.0 / (1.0 + np.log1p(pred_count))

        # Geometric mean of three signals
        geo_score = (sig1 * sig2 * sig3) ** (1/3)
        triple_scores.append(geo_score)

    if not triple_scores:
        return np.nan, 0
    return float(np.mean(triple_scores)), len(triple_scores)


In [16]:
novel_years = sorted(novel_edges['year'].unique())
print(f"\nNOVEL years to score: {novel_years}")

novel_results = []

for T in novel_years:
    print(f"\n{'='*50}")
    print(f"  Year T = {T}")

    # Training data: all SKG edges strictly before year T
    train_df = skg_edges[skg_edges['year'] < T].copy()
    print(f"  SKG training triples (year < {T}): {len(train_df):,}")

    if len(train_df) < 50:
        print(f"  ⚠ Skipping — too few training triples")
        continue

    # Build lookup tables from training data
    # 1. (predicate, entity) pair counts
    pe_count = train_df.groupby(['predicate', 'target']).size().to_dict()

    # 2. Entity → number of distinct predicates it appears with
    entity_pred_count = train_df.groupby('target')['predicate'].nunique().to_dict()

    # 3. Predicate → total count
    pred_count = train_df['predicate'].value_counts().to_dict()

    pe_paper_set = {
        'entity_pred_count': entity_pred_count,
        'pred_count':        pred_count
    }

    num_skg_papers  = train_df['source'].nunique()
    num_skg_triples = len(train_df)

    # Score NOVEL papers at year T
    novel_at_T = novel_edges[novel_edges['year'] == T]
    novel_papers = novel_at_T['source'].unique()
    print(f"  Scoring {len(novel_papers)} NOVEL papers")

    for paper_id in novel_papers:
        paper_triples = novel_at_T[novel_at_T['source'] == paper_id]
        score, n_triples = compute_pe_novelty(
            paper_triples, pe_count, pe_paper_set,
            num_skg_papers, num_skg_triples
        )
        novel_results.append({
            'paper_id':     paper_id,
            'transE_score': score,      # keeping column name for compatibility
            'pe_score':     score,
            'year':         T,
            'num_triples':  n_triples
        })

results_df = pd.DataFrame(novel_results).dropna(subset=['pe_score'])
print(f"\nTotal NOVEL papers scored: {len(results_df)}")


NOVEL years to score: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

  Year T = 2020
  SKG training triples (year < 2020): 169,904
  Scoring 4 NOVEL papers

  Year T = 2021
  SKG training triples (year < 2021): 234,348
  Scoring 404 NOVEL papers

  Year T = 2022
  SKG training triples (year < 2022): 234,348
  Scoring 23 NOVEL papers

  Year T = 2023
  SKG training triples (year < 2023): 298,455
  Scoring 86 NOVEL papers

  Year T = 2024
  SKG training triples (year < 2024): 366,526
  Scoring 9 NOVEL papers

  Year T = 2025
  SKG training triples (year < 2025): 377,828
  Scoring 2 NOVEL papers

Total NOVEL papers scored: 528


In [17]:
print("\n── Scoring SKG papers for validation ──")

# Full training data = all SKG
pe_count_full = skg_edges.groupby(['predicate','target']).size().to_dict()
entity_pred_count_full = skg_edges.groupby('target')['predicate'].nunique().to_dict()
pred_count_full = skg_edges['predicate'].value_counts().to_dict()

pe_paper_set_full = {
    'entity_pred_count': entity_pred_count_full,
    'pred_count':        pred_count_full
}

skg_results = []
for paper_id, grp in skg_edges.groupby('source'):
    score, n_triples = compute_pe_novelty(
        grp, pe_count_full, pe_paper_set_full,
        skg_edges['source'].nunique(), len(skg_edges)
    )
    skg_results.append({
        'paper_id':    paper_id,
        'pe_score':    score,
        'num_triples': n_triples
    })

skg_df = pd.DataFrame(skg_results).dropna(subset=['pe_score'])
print(f"SKG papers scored: {len(skg_df)}")



── Scoring SKG papers for validation ──
SKG papers scored: 2543


In [18]:
results_df_clean.to_csv(f'{BASE_PATH}/final/TransE_novelty_scores.csv', index=False)
print(f"Saved → {BASE_PATH}/final/TransE_novelty_scores.csv")
print(results_df_clean.head(10))

Saved → ../../outputs/final/TransE_novelty_scores.csv
       paper_id  transE_score  year  num_triples
0    NOVEL_MT_9     19.068465  2020           53
1   NOVEL_MT_95     20.177295  2020          181
2  NOVEL_MT_163     20.056812  2020          194
3   NOVEL_QA_38     19.667933  2020           84
4   NOVEL_DIA_0     21.925641  2021           99
5   NOVEL_DIA_1     21.223064  2021          132
6   NOVEL_DIA_2     20.642224  2021           94
7   NOVEL_DIA_3     21.479903  2021          206
8   NOVEL_DIA_4     22.206897  2021          206
9   NOVEL_DIA_5     21.593967  2021          103


In [19]:
novel_scores = results_df['pe_score'].values
skg_scores   = skg_df['pe_score'].values

stat, p = mannwhitneyu(novel_scores, skg_scores, alternative='greater')
r_eff = stat / (len(novel_scores) * len(skg_scores))

print(f"\n── Validation Results ──")
print(f"NOVEL mean: {novel_scores.mean():.4f}  (n={len(novel_scores)})")
print(f"SKG   mean: {skg_scores.mean():.4f}  (n={len(skg_scores)})")
print(f"Mann-Whitney U={stat:.1f},  p={p:.6f},  r={r_eff:.3f}")

if p < 0.05 and novel_scores.mean() > skg_scores.mean():
    print("✅ SUCCESS: NOVEL > SKG, p < 0.05")
else:
    print("❌ Failed — check output above")

# ── Per-year breakdown ────────────────────────────────────────────────────────
print("\n── Per-year NOVEL score breakdown ──")
print(results_df.groupby('year')['pe_score'].agg(['mean','std','count']))

# ── Domain breakdown ──────────────────────────────────────────────────────────
results_with_meta = results_df.merge(
    papers[['node_id','domain']], left_on='paper_id', right_on='node_id', how='left')
print("\n── Per-domain NOVEL score ──")
print(results_with_meta.groupby('domain')['pe_score'].agg(['mean','count']))


── Validation Results ──
NOVEL mean: 0.4550  (n=528)
SKG   mean: 0.2968  (n=2543)
Mann-Whitney U=1338012.0,  p=0.000000,  r=0.997
✅ SUCCESS: NOVEL > SKG, p < 0.05

── Per-year NOVEL score breakdown ──
          mean       std  count
year                           
2020  0.395719  0.020136      4
2021  0.414091  0.035923    404
2022  0.842177  0.181382     23
2023  0.536065  0.047217     86
2024  0.563188  0.101806      9
2025  0.405937  0.018791      2

── Per-domain NOVEL score ──
            mean  count
domain                 
DIA     0.493963    128
MT      0.450511    198
QA      0.419603     73
SA      0.453915     70
SUM     0.430410     59


In [25]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')


In [26]:
PROJ_DIM    = 100
EMBED_DIM   = 100
MARGIN      = 2.0
LR          = 0.01
EPOCHS      = 50
BATCH_SIZE  = 2048
NEG_SAMPLES = 10
HEAD_CORRUPT_PROB = 0.5   # 50% corrupt head, 50% corrupt tail
RNG         = np.random.default_rng(42)

In [22]:
BASE_PATH = '../../outputs'

EDGES_PATH = f'{BASE_PATH}/final/knowledge_edges.csv'
PAPERS_PATH = f'{BASE_PATH}/final/paper_nodes.csv'
EMB_PATH    = f'{BASE_PATH}/final/abstract_embeddings.npy'
IDS_PATH    = f'{BASE_PATH}/final/paper_ids.npy'


In [27]:
print("Loading files...")
edges   = pd.read_csv(EDGES_PATH)
papers  = pd.read_csv(PAPERS_PATH)
raw_emb = np.load(EMB_PATH)
raw_ids = np.load(IDS_PATH, allow_pickle=True)

edges = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)
paper_split = dict(zip(papers['node_id'], papers['split']))
paper_year  = dict(zip(papers['node_id'], papers['year']))
edges['split'] = edges['source'].map(paper_split)

skg_edges   = edges[edges['split'] == 'SKG'].copy()
novel_edges = edges[edges['split'] == 'NOVEL'].copy()
print(f"SKG: {len(skg_edges):,}  NOVEL: {len(novel_edges):,}")

Loading files...
SKG: 378,527  NOVEL: 82,599


In [28]:
print(f"\nPCA projection 768 → {PROJ_DIM}d...")
pca      = PCA(n_components=PROJ_DIM, random_state=42)
proj_emb = pca.fit_transform(raw_emb).astype(np.float32)
# Normalise once — embeddings stay frozen so this is their final form
proj_emb /= (np.linalg.norm(proj_emb, axis=1, keepdims=True) + 1e-9)
print(f"Variance explained: {100*pca.explained_variance_ratio_.sum():.1f}%")

id2emb   = {pid: proj_emb[i] for i, pid in enumerate(raw_ids)}

# Domain-specific fallback instead of global mean
# Papers missing abstracts get their domain's mean embedding
paper_domain = dict(zip(papers['node_id'], papers['domain']))
domain_means = {}
for pid, emb_vec in id2emb.items():
    d = paper_domain.get(pid, 'UNKNOWN')
    domain_means.setdefault(d, []).append(emb_vec)
domain_means = {d: np.mean(vecs, axis=0) for d, vecs in domain_means.items()}
for d in domain_means:
    domain_means[d] /= (np.linalg.norm(domain_means[d]) + 1e-9)
global_mean = proj_emb.mean(axis=0)
global_mean /= (np.linalg.norm(global_mean) + 1e-9)

def get_paper_emb(pid):
    if pid in id2emb:
        return id2emb[pid]
    d = paper_domain.get(pid, 'UNKNOWN')
    return domain_means.get(d, global_mean)


PCA projection 768 → 100d...
Variance explained: 85.5%


In [29]:
entity_vocab   = pd.unique(edges['target'])
relation_vocab = pd.unique(edges['predicate'])
entity2id      = {e: i for i, e in enumerate(entity_vocab)}
relation2id    = {r: i for i, r in enumerate(relation_vocab)}
NUM_E, NUM_R   = len(entity2id), len(relation2id)
print(f"Entities: {NUM_E:,}   Relations: {NUM_R:,}")

# Precompute all paper embeddings as a matrix for fast head corruption
all_paper_ids  = list(id2emb.keys())
all_paper_embs = np.array([id2emb[p] for p in all_paper_ids],
                           dtype=np.float32)  # [P, dim]
NUM_P = len(all_paper_ids)

Entities: 180,291   Relations: 18,494


In [30]:
class TransESciBERT:
    def __init__(self, num_e, num_r, dim):
        self.dim = dim
        bound    = 6.0 / np.sqrt(dim)
        self.E   = RNG.uniform(-bound, bound, (num_e, dim)).astype(np.float32)
        self.R   = RNG.uniform(-bound, bound, (num_r, dim)).astype(np.float32)
        # Init normalise
        self.E /= (np.linalg.norm(self.E, axis=1, keepdims=True) + 1e-9)
        self.R /= (np.linalg.norm(self.R, axis=1, keepdims=True) + 1e-9)

    def dist(self, h_emb, r_idx, t_idx):
        """
        Score triples. Embeddings must already be normalised before calling.
        Fix: NO normalisation inside this function.
        h_emb : [B, dim]  already-normalised head (paper SciBERT)
        r_idx : [B]
        t_idx : [B]
        Returns L1 distance [B]
        """
        R = self.R[r_idx]       # already normalised
        T = self.E[t_idx]       # already normalised
        return np.sum(np.abs(h_emb + R - T), axis=1)

    def train_epoch(self, h_embs, r_idx, t_idx,
                    all_eids, batch_size, margin, lr):
        N    = len(r_idx)
        perm = RNG.permutation(N)
        h_embs = h_embs[perm]
        r_idx  = r_idx[perm]
        t_idx  = t_idx[perm]

        total_loss = 0.0
        n_batches  = 0

        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            B   = end - start

            bh = h_embs[start:end]   # [B, dim] frozen SciBERT heads
            br = r_idx[start:end]
            bt = t_idx[start:end]

            R_emb = self.R[br]       # [B, dim]
            T_emb = self.E[bt]       # [B, dim]

            pos_d = np.sum(np.abs(bh + R_emb - T_emb), axis=1)  # [B]

            # ── Negative sampling: 50% head corrupt, 50% tail corrupt ─────────
            corrupt_head = RNG.random(B) < HEAD_CORRUPT_PROB  # [B] bool

            # Tail corruptions
            neg_t_ids = RNG.choice(all_eids, size=(B, NEG_SAMPLES))
            # Head corruptions (replace paper embedding with random paper emb)
            neg_h_idx = RNG.integers(0, NUM_P, size=(B, NEG_SAMPLES))
            neg_h_embs = all_paper_embs[neg_h_idx]  # [B, K, dim]

            # For each sample: use corrupted head OR corrupted tail
            neg_scores = np.zeros((B, NEG_SAMPLES), dtype=np.float32)

            for k in range(NEG_SAMPLES):
                mask = corrupt_head  # [B]

                # Tail-corrupted distance
                NT = self.E[neg_t_ids[:, k]]   # [B, dim]
                d_tail = np.sum(np.abs(bh + R_emb - NT), axis=1)

                # Head-corrupted distance
                NH = neg_h_embs[:, k, :]       # [B, dim]
                d_head = np.sum(np.abs(NH + R_emb - T_emb), axis=1)

                neg_scores[:, k] = np.where(mask, d_head, d_tail)

            # ── Loss: average over K negatives, then mean over batch ──────────
            # Fix: one loss value per triple, then mean over batch
            per_triple_loss = np.maximum(
                0.0, margin + pos_d[:, np.newaxis] - neg_scores
            ).mean(axis=1)                       # [B]
            batch_loss = per_triple_loss.mean()
            total_loss += batch_loss
            n_batches  += 1

            # ── Gradients ─────────────────────────────────────────────────────
            # Only update entity (tail) and relation embeddings
            # Head (paper) is frozen — no update
            active   = ((margin + pos_d[:, np.newaxis] - neg_scores) > 0
                        ).astype(np.float32)                 # [B, K]
            n_act    = active.sum(axis=1, keepdims=True) + 1e-9
            # Scale: average over active negatives, mean over batch
            scale_pos = (active / n_act).mean(axis=1, keepdims=True) / B  # [B,1]

            pos_sign = np.sign(bh + R_emb - T_emb)          # [B, dim]

            # Relation gradient
            self.R[br] -= lr * (scale_pos * pos_sign)

            # Positive tail gradient
            self.E[bt] -= lr * (-scale_pos * pos_sign)

            # Negative tail gradient (only tail-corrupted samples)
            tail_mask = (~corrupt_head).astype(np.float32)   # [B]
            for k in range(NEG_SAMPLES):
                act_k     = active[:, k] * tail_mask          # [B]
                scale_neg = (act_k / (n_act.squeeze() + 1e-9)
                             ).reshape(-1, 1) / (B * NEG_SAMPLES)
                NT_k      = self.E[neg_t_ids[:, k]]
                neg_sign  = np.sign(bh + R_emb - NT_k)
                grad_neg  = scale_neg * neg_sign
                np.add.at(self.E, neg_t_ids[:, k], lr * grad_neg)

            # ── Post-update normalisation (Fix: not inside dist()) ────────────
            # Normalise relation embeddings touched this batch
            self.R[br] /= (np.linalg.norm(
                self.R[br], axis=1, keepdims=True) + 1e-9)

            # Normalise entity embeddings touched this batch
            # Fix: avoid np.unique — just normalise bt and neg_t_ids directly
            self.E[bt] /= (np.linalg.norm(
                self.E[bt], axis=1, keepdims=True) + 1e-9)

            flat_neg = neg_t_ids.flatten()
            nt_norms = np.linalg.norm(
                self.E[flat_neg], axis=1, keepdims=True) + 1e-9
            self.E[flat_neg] /= nt_norms

            # L2 norm clipping: clip any entity that explodes > 1.5
            norms_bt = np.linalg.norm(self.E[bt], axis=1)
            clip_bt  = norms_bt > 1.5
            if clip_bt.any():
                self.E[bt[clip_bt]] /= norms_bt[clip_bt, np.newaxis]

        return total_loss / max(1, n_batches)

    def score_paper(self, paper_id, paper_triples):
        valid = (paper_triples['target'].isin(entity2id) &
                 paper_triples['predicate'].isin(relation2id))
        pt = paper_triples[valid]
        if len(pt) == 0:
            return np.nan, 0
        h_emb = get_paper_emb(paper_id)
        h_mat = np.tile(h_emb, (len(pt), 1))
        r_idx = pt['predicate'].map(relation2id).values.astype(np.int64)
        t_idx = pt['target'].map(entity2id).values.astype(np.int64)
        distances = self.dist(h_mat, r_idx, t_idx)
        return float(np.mean(distances)), len(pt)


In [31]:
def prepare_arrays(edge_df):
    valid = (edge_df['target'].isin(entity2id) &
             edge_df['predicate'].isin(relation2id))
    df = edge_df[valid].reset_index(drop=True)
    if len(df) == 0:
        return None, None, None
    h_embs = np.array([get_paper_emb(p) for p in df['source']],
                      dtype=np.float32)
    r_idx  = df['predicate'].map(relation2id).values.astype(np.int64)
    t_idx  = df['target'].map(entity2id).values.astype(np.int64)
    return h_embs, r_idx, t_idx

In [32]:
all_eids    = np.array(list(entity2id.values()), dtype=np.int64)
novel_years = sorted(novel_edges['year'].unique())
print(f"\nNOVEL years: {novel_years}")

novel_results = []

for T in novel_years:
    print(f"\n{'='*58}\n  T = {T}")
    train_df = skg_edges[skg_edges['year'] < T]
    print(f"  SKG training triples (year < {T}): {len(train_df):,}")
    if len(train_df) < 200:
        print("  ⚠ Skipping")
        continue

    h_tr, r_tr, t_tr = prepare_arrays(train_df)
    if h_tr is None:
        continue
    print(f"  Valid triples: {len(r_tr):,}")

    model = TransESciBERT(NUM_E, NUM_R, EMBED_DIM)

    for epoch in range(1, EPOCHS + 1):
        lr_t = LR if epoch <= 35 else LR * 0.3
        loss = model.train_epoch(
            h_tr, r_tr, t_tr, all_eids, BATCH_SIZE, MARGIN, lr_t)
        if epoch % 10 == 0:
            print(f"    Epoch {epoch:02d}/{EPOCHS}  "
                  f"loss={loss:.4f}  lr={lr_t:.4f}")

    novel_at_T     = novel_edges[novel_edges['year'] == T]
    novel_papers_T = novel_at_T['source'].unique()
    print(f"  Scoring {len(novel_papers_T)} NOVEL papers")

    for pid in novel_papers_T:
        pt    = novel_at_T[novel_at_T['source'] == pid]
        score, n = model.score_paper(pid, pt)
        novel_results.append({
            'paper_id':     pid,
            'transE_score': score,
            'year':         T,
            'num_triples':  n
        })



NOVEL years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

  T = 2020
  SKG training triples (year < 2020): 169,904
  Valid triples: 169,904
    Epoch 10/50  loss=1.9838  lr=0.0100
    Epoch 20/50  loss=1.9816  lr=0.0100
    Epoch 30/50  loss=1.9818  lr=0.0100
    Epoch 40/50  loss=1.9807  lr=0.0030
    Epoch 50/50  loss=1.9825  lr=0.0030
  Scoring 4 NOVEL papers

  T = 2021
  SKG training triples (year < 2021): 234,348
  Valid triples: 234,348
    Epoch 10/50  loss=1.9925  lr=0.0100
    Epoch 20/50  loss=1.9931  lr=0.0100
    Epoch 30/50  loss=1.9907  lr=0.0100
    Epoch 40/50  loss=1.9885  lr=0.0030
    Epoch 50/50  loss=1.9897  lr=0.0030
  Scoring 404 NOVEL papers

  T = 2022
  SKG training triples (year < 2022): 234,348
  Valid triples: 234,348
    Epoch 10/50  loss=1.9965  lr=0.0100
    Epoch 20/50  loss=1.9958  lr=0.0100
    Epoch 30/50  loss=1.9945  lr=0.0100
    Epoch 40/50  loss=1.9945  lr=0.0030
    Epoch 50/50  loss=1.9932

In [33]:
results_df = pd.DataFrame(novel_results).dropna(subset=['transE_score'])
print(f"\nScored {len(results_df)} NOVEL papers")
results_df.to_csv(f'{BASE_PATH}/final/transE_novelty_scores.csv', index=False)
print("Saved → transE_novelty_scores.csv")
print(results_df.describe())


Scored 528 NOVEL papers
Saved → transE_novelty_scores.csv
       transE_score         year  num_triples
count    528.000000   528.000000    528.00000
mean      13.450720  2021.428030    156.43750
std        0.464704     0.857233    173.36188
min       12.464311  2020.000000      4.00000
25%       13.036834  2021.000000     85.00000
50%       13.626709  2021.000000    134.00000
75%       13.785028  2021.000000    186.50000
max       14.361026  2025.000000   2592.00000


In [34]:
print("\n── Training validation model (all SKG) ──")
h_all, r_all, t_all = prepare_arrays(skg_edges)
model_val = TransESciBERT(NUM_E, NUM_R, EMBED_DIM)
for epoch in range(1, EPOCHS + 1):
    lr_t = LR if epoch <= 35 else LR * 0.3
    loss = model_val.train_epoch(
        h_all, r_all, t_all, all_eids, BATCH_SIZE, MARGIN, lr_t)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch}/{EPOCHS}  loss={loss:.4f}")

skg_results = []
for pid in skg_edges['source'].unique():
    pt = skg_edges[skg_edges['source'] == pid]
    score, n = model_val.score_paper(pid, pt)
    if not np.isnan(score):
        skg_results.append({
            'paper_id':     pid,
            'transE_score': score,
            'year':         paper_year.get(pid)
        })

skg_df = pd.DataFrame(skg_results)
print(f"SKG papers scored: {len(skg_df)}")


── Training validation model (all SKG) ──
  Epoch 10/50  loss=1.9798
  Epoch 20/50  loss=1.9778
  Epoch 30/50  loss=1.9768
  Epoch 40/50  loss=1.9761
  Epoch 50/50  loss=1.9744
SKG papers scored: 2543


In [35]:
ns = results_df['transE_score'].values
ss = skg_df['transE_score'].values

stat_g, p_g = mannwhitneyu(ns, ss, alternative='greater')
stat_l, p_l = mannwhitneyu(ns, ss, alternative='less')
r_g = stat_g / (len(ns) * len(ss))
r_l = stat_l / (len(ns) * len(ss))

print(f"\n{'='*58}")
print(f"── Validation Results ──")
print(f"NOVEL mean: {ns.mean():.4f}  std={ns.std():.4f}  n={len(ns)}")
print(f"SKG   mean: {ss.mean():.4f}  std={ss.std():.4f}  n={len(ss)}")
print(f"NOVEL > SKG:  p={p_g:.6f}  r={r_g:.3f}")
print(f"NOVEL < SKG:  p={p_l:.6f}  r={r_l:.3f}")

if p_g < 0.05 and ns.mean() > ss.mean():
    print("✅ SUCCESS: NOVEL > SKG, p < 0.05")
elif p_l < 0.05:
    print("⚠  INVERTED — NOVEL < SKG (invert scores in ensemble)")
else:
    print("❌ No significant separation")

print("\n── Per-year breakdown ──")
print(results_df.groupby('year')['transE_score'].agg(['mean','std','count']))

print("\n── Score vs triple count correlation ──")
print(f"r = {results_df['transE_score'].corr(results_df['num_triples']):.4f}")

results_meta = results_df.merge(
    papers[['node_id','domain']], left_on='paper_id', right_on='node_id', how='left')
print("\n── Per-domain NOVEL score ──")
print(results_meta.groupby('domain')['transE_score'].agg(['mean','std','count']))


── Validation Results ──
NOVEL mean: 13.4507  std=0.4643  n=528
SKG   mean: 13.6044  std=0.3524  n=2543
NOVEL > SKG:  p=1.000000  r=0.418
NOVEL < SKG:  p=0.000000  r=0.418
⚠  INVERTED — NOVEL < SKG (invert scores in ensemble)

── Per-year breakdown ──
           mean       std  count
year                            
2020  13.606785  0.220927      4
2021  13.417588  0.502277    404
2022  13.607905  0.265125     23
2023  13.560200  0.283691     86
2024  13.331690  0.355576      9
2025  13.851743  0.038517      2

── Score vs triple count correlation ──
r = 0.0177

── Per-domain NOVEL score ──
             mean       std  count
domain                            
DIA     13.558721  0.368219    128
MT      13.431838  0.474536    198
QA      13.410786  0.481190     73
SA      13.388022  0.504132     70
SUM     13.403578  0.524091     59


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, kruskal
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [2]:
BASE_PATH = '../../outputs'

EDGES_PATH  = f'{BASE_PATH}/final/knowledge_edges.csv'
PAPERS_PATH = f'{BASE_PATH}/final/paper_nodes.csv'
EMB_PATH    = f'{BASE_PATH}/final/abstract_embeddings.npy'
IDS_PATH    = f'{BASE_PATH}/final/paper_ids.npy'
MATRIX_PATH = f'{BASE_PATH}/final/novelty_feature_matrix_with_score.csv'
TRANSE_PATH = f'{BASE_PATH}/final/TransE_novelty_scores.csv'

In [3]:
PROJ_DIM    = 100
EMBED_DIM   = 100
MARGIN      = 2.0
LR          = 0.01
EPOCHS      = 50
BATCH_SIZE  = 2048
NEG_SAMPLES = 10
HEAD_CORRUPT_PROB = 0.5
RNG         = np.random.default_rng(42)

In [4]:
print("Loading files...")
edges   = pd.read_csv(EDGES_PATH)
papers  = pd.read_csv(PAPERS_PATH)
raw_emb = np.load(EMB_PATH)
raw_ids = np.load(IDS_PATH, allow_pickle=True)
matrix  = pd.read_csv(MATRIX_PATH)
transE_novel = pd.read_csv(TRANSE_PATH)  # NOVEL scores only

edges = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)
paper_split = dict(zip(papers['node_id'], papers['split']))
paper_year  = dict(zip(papers['node_id'], papers['year']))
edges['split'] = edges['source'].map(paper_split)

skg_edges   = edges[edges['split'] == 'SKG'].copy()
novel_edges = edges[edges['split'] == 'NOVEL'].copy()

Loading files...


In [7]:
print(f"PCA projection 768 → {PROJ_DIM}d...")
pca      = PCA(n_components=PROJ_DIM, random_state=42)
proj_emb = pca.fit_transform(raw_emb).astype(np.float32)
proj_emb /= (np.linalg.norm(proj_emb, axis=1, keepdims=True) + 1e-9)

id2emb = {pid: proj_emb[i] for i, pid in enumerate(raw_ids)}
paper_domain = dict(zip(papers['node_id'], papers['domain']))
domain_means = {}
for pid, ev in id2emb.items():
    d = paper_domain.get(pid, 'UNKNOWN')
    domain_means.setdefault(d, []).append(ev)
domain_means = {d: np.mean(v, axis=0) for d, v in domain_means.items()}
for d in domain_means:
    domain_means[d] /= (np.linalg.norm(domain_means[d]) + 1e-9)
global_mean = proj_emb.mean(axis=0)
global_mean /= (np.linalg.norm(global_mean) + 1e-9)

def get_paper_emb(pid):
    if pid in id2emb: return id2emb[pid]
    d = paper_domain.get(pid, 'UNKNOWN')
    return domain_means.get(d, global_mean)

all_paper_embs = proj_emb
NUM_P = len(proj_emb)

PCA projection 768 → 100d...


In [8]:
entity2id   = {e: i for i, e in enumerate(pd.unique(edges['target']))}
relation2id = {r: i for i, r in enumerate(pd.unique(edges['predicate']))}
NUM_E, NUM_R = len(entity2id), len(relation2id)
all_eids = np.array(list(entity2id.values()), dtype=np.int64)

In [9]:
class TransESciBERT:
    def __init__(self, num_e, num_r, dim):
        self.dim = dim
        bound    = 6.0 / np.sqrt(dim)
        self.E   = RNG.uniform(-bound, bound, (num_e, dim)).astype(np.float32)
        self.R   = RNG.uniform(-bound, bound, (num_r, dim)).astype(np.float32)
        self.E  /= (np.linalg.norm(self.E, axis=1, keepdims=True) + 1e-9)
        self.R  /= (np.linalg.norm(self.R, axis=1, keepdims=True) + 1e-9)

    def dist(self, h_emb, r_idx, t_idx):
        return np.sum(np.abs(h_emb + self.R[r_idx] - self.E[t_idx]), axis=1)

    def train_epoch(self, h_embs, r_idx, t_idx, batch_size, margin, lr):
        N    = len(r_idx)
        perm = RNG.permutation(N)
        h_embs, r_idx, t_idx = h_embs[perm], r_idx[perm], t_idx[perm]
        total_loss = 0.0; nb = 0

        for start in range(0, N, batch_size):
            end = min(start + batch_size, N); B = end - start
            bh = h_embs[start:end]; br = r_idx[start:end]; bt = t_idx[start:end]
            R_emb = self.R[br]; T_emb = self.E[bt]
            pos_d = np.sum(np.abs(bh + R_emb - T_emb), axis=1)

            corrupt_head = RNG.random(B) < HEAD_CORRUPT_PROB
            neg_t_ids  = RNG.choice(all_eids, (B, NEG_SAMPLES))
            neg_h_idx  = RNG.integers(0, NUM_P, (B, NEG_SAMPLES))
            neg_h_embs = all_paper_embs[neg_h_idx]
            neg_scores = np.zeros((B, NEG_SAMPLES), dtype=np.float32)

            for k in range(NEG_SAMPLES):
                NT = self.E[neg_t_ids[:, k]]
                NH = neg_h_embs[:, k, :]
                d_tail = np.sum(np.abs(bh + R_emb - NT), axis=1)
                d_head = np.sum(np.abs(NH + R_emb - T_emb), axis=1)
                neg_scores[:, k] = np.where(corrupt_head, d_head, d_tail)

            loss_mat = np.maximum(0.0, margin + pos_d[:, np.newaxis] - neg_scores)
            total_loss += loss_mat.mean(axis=1).mean(); nb += 1

            active  = (loss_mat > 0).astype(np.float32)
            n_act   = active.sum(axis=1, keepdims=True) + 1e-9
            scale   = (active / n_act).mean(axis=1, keepdims=True) / B
            ps      = np.sign(bh + R_emb - T_emb)
            self.R[br] -= lr * (scale * ps)
            self.E[bt] -= lr * (-scale * ps)

            tail_mask = (~corrupt_head).astype(np.float32)
            for k in range(NEG_SAMPLES):
                act_k     = active[:, k] * tail_mask
                scale_neg = (act_k / (n_act.squeeze()+1e-9)).reshape(-1,1)/(B*NEG_SAMPLES)
                NT_k      = self.E[neg_t_ids[:, k]]
                np.add.at(self.E, neg_t_ids[:, k],
                          lr * scale_neg * np.sign(bh + R_emb - NT_k))

            self.R[br] /= (np.linalg.norm(self.R[br], axis=1, keepdims=True)+1e-9)
            self.E[bt]  /= (np.linalg.norm(self.E[bt], axis=1, keepdims=True)+1e-9)
            flat = neg_t_ids.flatten()
            self.E[flat] /= (np.linalg.norm(self.E[flat], axis=1, keepdims=True)+1e-9)

        return total_loss / max(1, nb)

    def score_paper(self, pid, paper_triples):
        valid = (paper_triples['target'].isin(entity2id) &
                 paper_triples['predicate'].isin(relation2id))
        pt = paper_triples[valid]
        if len(pt) == 0: return np.nan, 0
        h_emb = get_paper_emb(pid)
        h_mat = np.tile(h_emb, (len(pt), 1))
        r_idx = pt['predicate'].map(relation2id).values.astype(np.int64)
        t_idx = pt['target'].map(entity2id).values.astype(np.int64)
        return float(np.mean(self.dist(h_mat, r_idx, t_idx))), len(pt)


def prepare_arrays(edge_df):
    valid = edge_df['target'].isin(entity2id) & edge_df['predicate'].isin(relation2id)
    df = edge_df[valid].reset_index(drop=True)
    if len(df) == 0: return None, None, None
    h = np.array([get_paper_emb(p) for p in df['source']], dtype=np.float32)
    r = df['predicate'].map(relation2id).values.astype(np.int64)
    t = df['target'].map(entity2id).values.astype(np.int64)
    return h, r, t

In [10]:
print("\n── Training validation model on all SKG ──")
h_all, r_all, t_all = prepare_arrays(skg_edges)
model_val = TransESciBERT(NUM_E, NUM_R, EMBED_DIM)

for epoch in range(1, EPOCHS + 1):
    lr_t = LR if epoch <= 35 else LR * 0.3
    loss = model_val.train_epoch(h_all, r_all, t_all, BATCH_SIZE, MARGIN, lr_t)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:02d}/{EPOCHS}  loss={loss:.4f}")

# Score ALL papers (NOVEL + SKG) for ensemble
print("\n── Scoring ALL papers ──")
all_results = []

# Score NOVEL papers
for pid in novel_edges['source'].unique():
    pt = novel_edges[novel_edges['source'] == pid]
    sc, n = model_val.score_paper(pid, pt)
    all_results.append({
        'paper_id': pid, 'transE_score': sc,
        'split': 'NOVEL', 'year': paper_year.get(pid)
    })

# Score SKG papers
for pid in skg_edges['source'].unique():
    pt = skg_edges[skg_edges['source'] == pid]
    sc, n = model_val.score_paper(pid, pt)
    all_results.append({
        'paper_id': pid, 'transE_score': sc,
        'split': 'SKG', 'year': paper_year.get(pid)
    })

scores_df = pd.DataFrame(all_results).dropna(subset=['transE_score'])
print(f"Total papers scored: {len(scores_df)}")
print(scores_df['split'].value_counts())


── Training validation model on all SKG ──
  Epoch 10/50  loss=1.9814
  Epoch 20/50  loss=1.9797
  Epoch 30/50  loss=1.9778
  Epoch 40/50  loss=1.9763
  Epoch 50/50  loss=1.9765

── Scoring ALL papers ──
Total papers scored: 3071
split
SKG      2543
NOVEL     528
Name: count, dtype: int64


In [11]:
scores_df[['paper_id','transE_score','year']].to_csv(
    f'{BASE_PATH}/final/transE_all_scores.csv', index=False)
print("Saved → transE_all_scores.csv")

Saved → transE_all_scores.csv


In [12]:
# Quick check: separation before ensemble
nov_s = scores_df[scores_df['split']=='NOVEL']['transE_score'].values
skg_s = scores_df[scores_df['split']=='SKG']['transE_score'].values
stat,p = mannwhitneyu(nov_s, skg_s, alternative='less')  # expect NOVEL < SKG (inverted)
r = stat/(len(nov_s)*len(skg_s))
print(f"\nTransE raw — NOVEL mean={nov_s.mean():.4f}  SKG mean={skg_s.mean():.4f}")
print(f"NOVEL<SKG: p={p:.6f}  r={r:.3f}  {'✅ inverted as expected' if p<0.05 else '❌'}")


TransE raw — NOVEL mean=13.4581  SKG mean=13.6130
NOVEL<SKG: p=0.000000  r=0.418  ✅ inverted as expected


In [13]:
print("\n── Building ensemble ──")
matrix = pd.read_csv(MATRIX_PATH)
matrix = matrix.merge(scores_df[['paper_id','transE_score']], on='paper_id', how='left')
matrix = matrix.merge(papers[['node_id','split','domain']],
                      left_on='paper_id', right_on='node_id', how='left')

novel_mask = matrix['split'] == 'NOVEL'
skg_mask   = matrix['split'] == 'SKG'
blog_mask  = matrix['split'] == 'BLOG'

# ── Invert TransE: max - score → high = novel ─────────────────────────────────
max_t = matrix['transE_score'].max()
matrix['transE_novelty'] = max_t - matrix['transE_score']

# Fill remaining NaN (papers with no edges) with domain mean
for domain in matrix['domain'].unique():
    dm = matrix['domain'] == domain
    fill = matrix.loc[dm & matrix['transE_novelty'].notna(), 'transE_novelty'].mean()
    if not np.isnan(fill):
        matrix.loc[dm & matrix['transE_novelty'].isna(), 'transE_novelty'] = fill
# Global fallback
global_fill = matrix['transE_novelty'].mean()
matrix['transE_novelty'] = matrix['transE_novelty'].fillna(global_fill)

# ── Semantic signal: lower similarity = more novel = use raw sem_knn inverted ─
matrix['sem_novelty'] = 1 - matrix['semantic_knn']



── Building ensemble ──


In [14]:
def mm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn + 1e-9)

matrix['transE_norm'] = mm(matrix['transE_novelty'])
matrix['sem_norm']    = mm(matrix['sem_novelty'])

In [15]:
print("\n── Individual signal check ──")
for col, label in [('transE_norm', 'TransE (inverted+normed)'),
                   ('sem_norm',    'Semantic (inverted+normed)')]:
    n = matrix.loc[novel_mask, col].values
    s = matrix.loc[skg_mask,   col].values
    st, p = mannwhitneyu(n, s, alternative='greater')
    r = st/(len(n)*len(s))
    print(f"  {label:35s}: NOVEL={n.mean():.4f} SKG={s.mean():.4f} "
          f"p={p:.4f} r={r:.3f} {'✅' if p<0.05 and n.mean()>s.mean() else '❌'}")


── Individual signal check ──
  TransE (inverted+normed)           : NOVEL=0.5514 SKG=0.5079 p=0.0000 r=0.575 ✅
  Semantic (inverted+normed)         : NOVEL=0.1626 SKG=0.1611 p=0.9666 r=0.478 ❌


In [16]:
print("\n── Grid Search ──")
print(f"{'α_transE':>10} {'NOVEL':>8} {'SKG':>8} {'p':>10} {'r':>7} {'OK':>4}")

best_r, best_alpha = -1, 0.0
for alpha in np.arange(0.0, 1.01, 0.05):
    score = alpha * matrix['transE_norm'] + (1-alpha) * matrix['sem_norm']
    n = score[novel_mask].values; s = score[skg_mask].values
    st, p = mannwhitneyu(n, s, alternative='greater')
    r = st/(len(n)*len(s))
    ok = p < 0.05 and n.mean() > s.mean()
    print(f"{alpha:>10.2f} {n.mean():>8.4f} {s.mean():>8.4f} "
          f"{p:>10.6f} {r:>7.3f} {'✅' if ok else '❌'}")
    if ok and r > best_r:
        best_r = r; best_alpha = alpha

print(f"\nBest: α_transE={best_alpha:.2f}  α_sem={1-best_alpha:.2f}  r={best_r:.4f}")


── Grid Search ──
  α_transE    NOVEL      SKG          p       r   OK
      0.00   0.1626   0.1611   0.966621   0.478 ❌
      0.05   0.1820   0.1784   0.983909   0.475 ❌
      0.10   0.2014   0.1958   0.982904   0.475 ❌
      0.15   0.2209   0.2131   0.900071   0.485 ❌
      0.20   0.2403   0.2305   0.033895   0.521 ✅
      0.25   0.2598   0.2478   0.000000   0.560 ✅
      0.30   0.2792   0.2651   0.000000   0.580 ✅
      0.35   0.2986   0.2825   0.000000   0.590 ✅
      0.40   0.3181   0.2998   0.000000   0.596 ✅
      0.45   0.3375   0.3172   0.000000   0.599 ✅
      0.50   0.3570   0.3345   0.000000   0.600 ✅
      0.55   0.3764   0.3518   0.000000   0.601 ✅
      0.60   0.3959   0.3692   0.000000   0.600 ✅
      0.65   0.4153   0.3865   0.000000   0.599 ✅
      0.70   0.4347   0.4038   0.000000   0.597 ✅
      0.75   0.4542   0.4212   0.000000   0.594 ✅
      0.80   0.4736   0.4385   0.000000   0.591 ✅
      0.85   0.4931   0.4559   0.000000   0.587 ✅
      0.90   0.5125   0.4732

In [17]:
matrix['final_novelty_score'] = (
    best_alpha * matrix['transE_norm'] +
    (1-best_alpha) * matrix['sem_norm']
)


In [18]:
nov_f = matrix.loc[novel_mask, 'final_novelty_score'].values
skg_f = matrix.loc[skg_mask,   'final_novelty_score'].values
blg_f = matrix.loc[blog_mask,  'final_novelty_score'].values

st_b, p_b = mannwhitneyu(nov_f, skg_f, alternative='greater')
r_b = st_b/(len(nov_f)*len(skg_f))

print(f"\n{'='*58}")
print(f"── Final Validation ──")
print(f"NOVEL mean: {nov_f.mean():.4f}  n={len(nov_f)}")
print(f"SKG   mean: {skg_f.mean():.4f}  n={len(skg_f)}")
print(f"BLOG  mean: {blg_f.mean():.4f}  n={len(blg_f)}")
print(f"Mann-Whitney: p={p_b:.6f}  r={r_b:.4f}")
print("✅ SUCCESS" if (p_b<0.05 and nov_f.mean()>skg_f.mean()) else "❌ Failed")

# 3-way test
st_kw, p_kw = kruskal(nov_f, skg_f, blg_f)
print(f"\n3-way Kruskal-Wallis: KW={st_kw:.1f}  p={p_kw:.6f}")

# Ablation
print("\n── Ablation ──")
for sig, label in [('transE_norm', 'TransE only'),
                   ('sem_norm',    'Semantic only'),
                   ('final_novelty_score', f'Ensemble α={best_alpha:.2f}')]:
    n = matrix.loc[novel_mask,sig].values
    s = matrix.loc[skg_mask,sig].values
    st, p = mannwhitneyu(n,s,alternative='greater')
    r = st/(len(n)*len(s))
    print(f"  {label:30s}: p={p:.6f}  r={r:.3f}  "
          f"NOVEL={n.mean():.4f} SKG={s.mean():.4f}  "
          f"{'✅' if p<0.05 and n.mean()>s.mean() else '❌'}")

print("\n── Per-domain (NOVEL) ──")
print(matrix[novel_mask].groupby('domain')['final_novelty_score']
      .agg(['mean','std','count']).sort_values('mean', ascending=False))

print("\n── Per-year ──")
yr = matrix[novel_mask|skg_mask].groupby(['year','split'])['final_novelty_score'].mean().unstack()
print(yr.dropna(how='all').to_string())



── Final Validation ──
NOVEL mean: 0.3764  n=762
SKG   mean: 0.3518  n=2916
BLOG  mean: 0.3897  n=564
Mann-Whitney: p=0.000000  r=0.6005
✅ SUCCESS

3-way Kruskal-Wallis: KW=332.9  p=0.000000

── Ablation ──
  TransE only                   : p=0.000000  r=0.575  NOVEL=0.5514 SKG=0.5079  ✅
  Semantic only                 : p=0.966621  r=0.478  NOVEL=0.1626 SKG=0.1611  ❌
  Ensemble α=0.55               : p=0.000000  r=0.601  NOVEL=0.3764 SKG=0.3518  ✅

── Per-domain (NOVEL) ──
            mean       std  count
domain                           
MT      0.385418  0.080574    258
SA      0.384925  0.087880    111
QA      0.378107  0.084045     88
SUM     0.370541  0.102430     65
DIA     0.363754  0.055206    240

── Per-year ──
split     NOVEL       SKG
year                     
2010        NaN  0.253351
2011        NaN  0.341464
2012        NaN  0.343722
2013        NaN  0.330436
2014        NaN  0.329947
2015        NaN  0.398877
2016        NaN  0.334098
2017        NaN  0.333093
2018  

In [19]:
drop_cols = ['node_id','split','domain','sem_novelty',
             'transE_novelty','sem_norm','transE_norm']
save_df = matrix.drop(columns=[c for c in drop_cols if c in matrix.columns])
save_df.to_csv(f'{BASE_PATH}/intermediate/novelty_feature_matrix_with_score.csv', index=False)
print(f"\nSaved → novelty_feature_matrix_with_score.csv")
print(f"Shape: {save_df.shape}")
print(f"Columns: {save_df.columns.tolist()}")


Saved → novelty_feature_matrix_with_score.csv
Shape: (4242, 14)
Columns: ['paper_id', 'year', 'semantic_knn', 'structural_novelty', 'triple_count', 'cited_by_count', 'reference_count', 'citation_signal', 'struct_norm', 'semantic_norm', 'citation_norm', 'composite_novelty', 'transE_score', 'final_novelty_score']
